# ViANLI-RL: SFT + GRPO với Suy Luận Căn Chỉnh Tường Minh (Explicit Alignment)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baonguyen-infina/rl-nli-/blob/master/notebooks/g0_setup.ipynb)

Notebook này tập trung huấn luyện **phương pháp chính** của nghiên cứu:
- **E5**: SFT với định dạng căn chỉnh tường minh (`align`).
- **E11**: SFT $\rightarrow$ GRPO (`E5` $\rightarrow$ `E11`) tối ưu hàm thưởng căn chỉnh $R = r_{\text{acc}} + 0.2 r_{\text{fmt}} + 0.3 r_{\text{acc}} r_{\text{struct}}$.
- **E8**: Pure GRPO trực tiếp trên định dạng căn chỉnh từ mô hình nền `Qwen/Qwen3.5-2B`.

### Trình tự thực hiện:
1. **Thiết lập môi trường**: Kiểm tra GPU, clone repo, gỡ `torchao` cũ, cài thư viện.
2. **Dữ liệu & Test**: Chuẩn bị dataset ViANLI, kiểm tra unit tests (85 tests).
3. **Huấn luyện Phương Pháp Chính**:
   - **Bước 1**: Huấn luyện SFT Align (`E5`).
   - **Bước 2**: Huấn luyện SFT $\rightarrow$ GRPO Align (`E11`).
   - **Bước 3 (Đối chứng)**: Huấn luyện Pure GRPO Align (`E8`).
4. **So sánh kết quả**: Bảng đối chiếu Accuracy, Macro F1, Parse Rate so với các baselines (`E0`..`E3`).
5. **Sao lưu**: Backup kết quả và checkpoints vào Google Drive.

## 1. Thiết lập Môi trường & Dependencies

In [ ]:
# 1.1 Kiểm tra thông tin GPU
!nvidia-smi

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 1.2 Mount Google Drive (tùy chọn - để lưu checkpoints và kết quả)
import os
USE_DRIVE = True
drive_backup_dir = "/content/drive/MyDrive/rl-nli_backup"

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(f"{drive_backup_dir}/results", exist_ok=True)
        os.makedirs(f"{drive_backup_dir}/artifacts", exist_ok=True)
        print(f"✅ Đã mount Drive thành công: {drive_backup_dir}")
    except Exception as e:
        print(f"⚠️ Bỏ qua mount Drive: {e}")

In [ ]:
# 1.3 Clone hoặc cập nhật repository (đảm bảo đồng bộ 100% với commit mới nhất)
import os
repo_url = "https://github.com/baonguyen-infina/rl-nli-.git"
repo_dir = "/content/rl-nli"

if not os.path.exists(repo_dir):
    !git clone {repo_url} {repo_dir}
else:
    %cd {repo_dir}
    !git fetch origin master && git reset --hard origin/master

%cd {repo_dir}


In [ ]:
# 1.4 Cài đặt các thư viện cần thiết (chỉ cài những gói Colab chưa có sẵn, tránh tốn thời gian)
!pip install -q trl==1.15.0 vllm peft datasets


## 2. Chuẩn bị Dữ liệu & Kiểm tra Hệ Thống

In [ ]:
# 2.1 Chuẩn hoá dataset ViANLI và sinh config cho ma trận thử nghiệm
!python -m data.prepare
!python -m configs.matrix

# 2.2 Chuẩn bị rationales teacher cho format align và cot (mất ~1 giây)
!python -m teacher.generate --format align --backend heuristic
!python -m teacher.generate --format cot --backend heuristic


In [ ]:
# 2.3 Chạy bộ test suite (85 tests - đảm bảo parser, reward, grounding, datasets chạy chính xác)
!python -m pytest -q

## 3. HUẤN LUYỆN PHƯƠNG PHÁP CHÍNH (MAIN METHOD)

### Trọng tâm nghiên cứu: Căn chỉnh tường minh (`align` format)
- Output của mô hình có dạng:
```
<align>
H1: "<đoạn hypothesis>" <-> "<đoạn premise>" | match
H2: "<đoạn hypothesis>" <-> ∅ | missing
H3: "<đoạn hypothesis>" <-> "<đoạn premise>" | conflict
</align>
<answer>nhãn</answer>
```
- Hàm thưởng RL: $R = r_{\text{acc}} + 0.2 r_{\text{fmt}} + 0.3 r_{\text{acc}} r_{\text{struct}}$.
- Điểm mấu chốt: Buộc LLM phải tìm bằng chứng trích dẫn nguyên văn (`is_grounded`) và tuân thủ luật suy luận logic tự nhiên để vượt qua bẫy suy diễn của tập đối nghịch ViANLI.

### 3.1 Bước 1: Huấn luyện SFT Căn Chỉnh (`E5`)
Huấn luyện Qwen3.5-2B LoRA trong 3 epochs trên tập ViANLI với định dạng `align`.
Mỗi epoch sẽ tự động đánh giá trên tập dev và lưu adapter tốt nhất tại `results/E5_Qwen3.5-2B_s0/best_adapter`.

In [ ]:
# 3.1 Huấn luyện SFT Căn Chỉnh (E5)
# (Nếu muốn chạy thử nghiệm SFT Label E3 trước: đổi E5 thành E3)
!python -m scripts.run --run E5 --seed 0


### 3.2 Bước 2: Huấn luyện SFT $\rightarrow$ GRPO Căn Chỉnh (`E11`)
Khởi tạo từ checkpoint tốt nhất của `E5`, sau đó chạy 500 bước GRPO RL với vLLM rollout và hàm thưởng căn chỉnh.
Sau khi hoàn thành, tự động chạy greedy decoding trên dev split và ghi nhận kết quả.

In [ ]:
# Huấn luyện SFT -> GRPO Align (E11 - Phương pháp chính hoàn chỉnh)
!python -m scripts.run --run E11 --seed 0

### 3.3 Bước 3 (Đối chứng): Huấn luyện Pure GRPO Căn Chỉnh (`E8`)
Chạy GRPO trực tiếp từ mô hình nền `Qwen/Qwen3.5-2B` (không qua bước SFT) để đo lường khả năng tự khám phá cấu trúc căn chỉnh của thuật toán RL.

In [ ]:
# Huấn luyện Pure GRPO Align (E8)
!python -m scripts.run --run E8 --seed 0

## 4. (Tùy chọn) Chạy các Thử nghiệm Đối chứng Khác
Dùng để bổ sung số liệu cho bảng so sánh trong bài báo.

In [ ]:
# Zero-shot baselines (E0: label, E1: cot, E2: align)
# !python -m scripts.run --run E0 --seed 0
# !python -m scripts.run --run E1 --seed 0
# !python -m scripts.run --run E2 --seed 0

# SFT label baseline (E3)
# !python -m scripts.run --run E3 --seed 0

# GRPO label (E6) & GRPO CoT (E7)
# !python -m scripts.run --run E6 --seed 0
# !python -m scripts.run --run E7 --seed 0

## 5. Tổng Hợp & So Sánh Kết Quả
Bảng tổng hợp metrics trên tập dev của tất cả các run đã thực hiện.

In [ ]:
import json
from pathlib import Path

results_dir = Path("results")
print(f"{'Run ID':<10} | {'Model / Description':<25} | {'Accuracy':<10} | {'Macro F1':<10} | {'Parse Rate':<12}")
print("-" * 78)

DESCRIPTIONS = {
    "E0": "Zero-Shot Label",
    "E1": "Zero-Shot CoT",
    "E2": "Zero-Shot Align",
    "E3": "SFT Label",
    "E4": "SFT CoT",
    "E5": "SFT Align",
    "E6": "GRPO Label",
    "E7": "GRPO CoT",
    "E8": "Pure GRPO Align",
    "E9": "SFT -> GRPO Label",
    "E10": "SFT -> GRPO CoT",
    "E11": "★ SFT -> GRPO Align (MAIN)",
}

for m_file in sorted(results_dir.glob("*/metrics_dev.json")):
    run_name = m_file.parent.name
    run_id = run_name.split("_")[0]
    desc = DESCRIPTIONS.get(run_id, run_name)
    with open(m_file, "r", encoding="utf-8") as f:
        m = json.load(f)
    acc = f"{m.get('accuracy', 0):.4f}"
    f1 = f"{m.get('macro_f1', 0):.4f}"
    parse = f"{m.get('parse_rate', 0):.4f}" if "parse_rate" in m else "N/A"
    print(f"{run_id:<10} | {desc:<25} | {acc:<10} | {f1:<10} | {parse:<12}")

## 6. Sao Lưu Vào Google Drive

In [ ]:
!cp -ru /content/rl-nli/results /content/drive/MyDrive/rl-nli_backup/ 2>/dev/null && echo "✅ Đã backup results sang Google Drive!" || echo "(Không backup do chưa mount Drive)"
!cp -ru /content/rl-nli/artifacts /content/drive/MyDrive/rl-nli_backup/ 2>/dev/null && echo "✅ Đã backup artifacts sang Google Drive!" || true